# Evaluación — Módulo de Interpolación y Aproximación

**Programación Científica 2026-1 · Universidad Nacional de Colombia**

mbastidaso@unal.edu.co


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

**Integrantes:** (orden alfabético)
- Apellido1, Nombre1: `Ospina, Juan`
- Apellido2, Nombre2: `Serrano, Gabriel`

---

## Instrucciones generales

**Duración:** 1.5 horas

**Recursos permitidos:** internet, LLMs, notebooks del curso. No pueden compartir código ni respuestas con otras parejas.

**Entrega — flujo GitHub obligatorio:**

0. Crear una copia de este archivo que se llama `apellido1_apellido2_evaluacion1.ipynb` en la carpeta EVALUACION y sobre ese archivo se harán todos los resultados/cambios/commits.

1. Crear una rama con el nombre exacto: `apellido1_apellido2_5Junio` (orden alfabético, minúsculas, sin tildes, sin espacios).

2. Trabajen **únicamente sobre esa rama**. Deben hacer **exactamente 3 commits** antes de abrir el PR.

3. Los mensajes de commit deben seguir esta convención y cubrir avance real:
   ```
   init: cedulas, seed y generacion de datos
   add: interpolacion spline y ajuste LS
   add: PCA y analisis escritos
   ```

4. Abran un **Pull Request** de su rama a `main`. En el título del PR escriban: `Evaluación [Apellido1] [Apellido2]`.

⚠️ **Penalizaciones automáticas:**
- Notebook que no ejecuta de principio a fin (`Kernel → Restart & Run All`): −10 pts.
- Menos o más de 3 commits: −5 pts.
- Rama con nombre incorrecto: −5 pts.

---

**Distribución de puntos:**

| Sección | Puntos |
|---|---|
| P0 — Inicialización y datos | 5 |
| P1 — Cuestionario conceptual | 20 |
| P2 — Interpolación B-spline 2D | 30 |
| P3 — Ajuste por mínimos cuadrados | 20 |
| P4 — PCA | 15 |
| GitHub (commits + PR + respuesta) | 10 |
| **Total** | **100** |

---
## P0 — Inicialización (5 pts)

Escriban sus números de cédula en las variables indicadas. **No modifiquen ninguna otra línea de esta sección.** Todos los datos del ejercicio se generan a partir de su semilla.

In [ ]:
# TODO P0: reemplacen los ceros con sus ID reales
cedula1 = 1001755818
cedula2 = 1026308885

In [ ]:
# 🔒 CELDA DE DATOS — no modificar
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RectBivariateSpline
from scipy.linalg import svd

assert cedula1 != 0 and cedula2 != 0, "Deben ingresar sus ID en la celda anterior."

seed = int((cedula1 + cedula2) % 1000)
rng  = np.random.default_rng(seed)

# Parámetros de la superficie — dependen de la semilla
alpha = 1.0 + (seed % 7)  * 0.3          # frecuencia en x: entre 1.0 y 2.8
beta  = 1.0 + (seed % 5)  * 0.4          # frecuencia en y: entre 1.0 y 2.6
gamma = 0.10 + (seed % 11) * 0.02        # decaimiento gaussiano: entre 0.10 y 0.30
noise_std = 0.05 + (seed % 13) * 0.01    # ruido: entre 0.05 y 0.17

# Malla gruesa de muestreo (15x15 puntos)
n  = 15
x_raw = np.linspace(-3, 3, n)
y_raw = np.linspace(-3, 3, n)
X_raw, Y_raw = np.meshgrid(x_raw, y_raw, indexing='ij')

# Elevación verdadera + ruido
Z_true = np.sin(alpha * X_raw) * np.cos(beta * Y_raw) * np.exp(-gamma * (X_raw**2 + Y_raw**2))
Z_data = Z_true + rng.normal(0, noise_std, X_raw.shape)

print(f"Semilla: {seed}")
print(f"Parámetros — alpha: {alpha:.2f}, beta: {beta:.2f}, gamma: {gamma:.3f}, ruido: {noise_std:.3f}")
print(f"Malla de datos: {n}x{n} puntos en [-3,3]x[-3,3]")
print(f"Z_data shape: {Z_data.shape}")

Semilla: 703
Parámetros — alpha: 1.90, beta: 2.20, gamma: 0.300, ruido: 0.060
Malla de datos: 15x15 puntos en [-3,3]x[-3,3]
Z_data shape: (15, 15)


---
## P1 — Cuestionario conceptual (20 pts)

Respondan las siguientes preguntas **sin ejecutar código adicional**. Basen sus respuestas en la lectura del bloque de generación de datos de P0.

### P1.1 (7 pts)

La celda de datos genera `Z_data = Z_true + ruido`. Cuando construyan el spline en P2, este va a **interpolar** `Z_data`, no `Z_true`.

**a)** ¿Qué significa que el spline interpole `Z_data` en lugar de ajustar `Z_true`? ¿El spline va a pasar exactamente por los valores de `Z_data`?  
**b)** ¿En qué condición sería preferible usar mínimos cuadrados en lugar del spline para reconstruir la superficie?

**Respuesta P1.1:**

**a) ¿Qué significa que el spline interpole Z_data en lugar de ajustar Z_true? ¿El spline va a pasar exactamente por los valores de Z_data?**

La superficie de datos se genera como $Z_{data} = Z_{true} + \text{ruido}$, por lo que los valores observados contienen tanto la señal verdadera como errores o perturbaciones aleatorias. Al construir un spline interpolante sobre $Z_{data}$, el spline utiliza directamente los datos observados y no la superficie verdadera $Z_{true}$, la cual es desconocida.

Como se trata de una interpolación, el spline pasa exactamente por todos los puntos de la malla $(x_i, y_j, Z_{data,ij})$. En consecuencia, reproduce tanto la tendencia de la superficie como el ruido presente en las mediciones.

**b) ¿En qué condición sería preferible usar mínimos cuadrados en lugar del spline para reconstruir la superficie?**

Sería preferible utilizar mínimos cuadrados cuando los datos contengan ruido significativo o errores de medición y se busque recuperar la tendencia subyacente de la superficie en lugar de reproducir exactamente cada observación. En este caso, un ajuste por mínimos cuadrados puede suavizar el efecto del ruido, mientras que un spline interpolante lo incorpora completamente al pasar por todos los datos observados.


### P1.2 (7 pts)

En P3 van a ajustar un plano $z = c_0 + c_1 x + c_2 y$ a los datos usando mínimos cuadrados.

**a)** ¿Cómo se construye la matriz $A$ del sistema $A\mathbf{c} = \mathbf{z}$ para este modelo? Descríbanla en términos de sus columnas.  
**b)** La función verdadera `Z_true` no es un plano. ¿Qué representa entonces el plano ajustado? ¿Qué información útil les da sobre la superficie?

**Respuesta P1.2:**

**a) ¿Cómo se construye la matriz $A$ del sistema $Ac = z$ para este modelo? Descríbanla en términos de sus columnas.**

Para el modelo lineal $z = c_0 + c_1 x + c_2 y$, cada observación $(x_i, y_i, z_i)$ genera una ecuación:

$$z_i = c_0 + c_1 x_i + c_2 y_i$$

Por tanto, la matriz $A$ se construye colocando una fila por cada observación:

$$A = \begin{bmatrix} 1 & x_1 & y_1 \\ 1 & x_2 & y_2 \\ \vdots & \vdots & \vdots \\ 1 & x_N & y_N \end{bmatrix}$$

La primera columna contiene unos (correspondientes al término independiente $c_0$), la segunda contiene las coordenadas $x_i$ y la tercera las coordenadas $y_i$. El vector de incógnitas es $c = [c_0, c_1, c_2]^T$ y el vector de observaciones es $z = [z_1, z_2, \dots, z_N]^T$.

**b) La función verdadera Z_true no es un plano. ¿Qué representa entonces el plano ajustado? ¿Qué información útil les da sobre la superficie?**

Como $Z_{true}$ no es un plano, el modelo $z = c_0 + c_1 x + c_2 y$ no puede reproducir exactamente la superficie verdadera. El plano ajustado por mínimos cuadrados representa la aproximación lineal que mejor se adapta globalmente a los datos en el sentido de minimizar la suma de los cuadrados de los residuos.

Este plano proporciona información sobre la tendencia general de la superficie, permitiendo identificar cómo varía en promedio $z$ con respecto a $x$ y $y$. Aunque no captura curvaturas ni detalles locales, resume el comportamiento global de la superficie mediante una descripción simple y fácil de interpretar.


### P1.3 (6 pts)

En P4 van a aplicar PCA a la nube de puntos $(x_i, y_i, z_i)$ formada por los 225 puntos de la malla.

**a)** ¿Por qué es necesario centrar los datos antes de calcular los componentes principales?  
**b)** ¿Qué interpretación geométrica tiene el primer componente principal en el contexto de esta superficie?

**Respuesta P1.3:**

**a) ¿Por qué es necesario centrar los datos antes de calcular los componentes principales?**

Es necesario centrar los datos antes de calcular los componentes principales porque PCA busca las direcciones de máxima variabilidad con respecto al promedio de los datos. Al restar la media de cada variable, el análisis se realiza sobre las fluctuaciones alrededor del centro de la nube de puntos y no sobre su posición absoluta en el espacio. Si los datos no se centran, los componentes principales pueden verse influenciados por el desplazamiento de la nube y no representar correctamente las direcciones de mayor variación.

**b) ¿Qué interpretación geométrica tiene el primer componente principal en el contexto de esta superficie?**

Geométricamente, el primer componente principal corresponde a la dirección en la que la nube de puntos presenta la mayor dispersión. En el contexto de esta superficie, representa la dirección espacial a lo largo de la cual los puntos $(x_i, y_i, z_i)$ varían más. Por tanto, resume la tendencia dominante de la forma de la superficie y proporciona la mejor aproximación lineal unidimensional a la nube de datos en el sentido de máxima varianza.

---
**Recordatorio del flujo:**

```bash
# 1. Crear y moverse a su rama
git checkout -b apellido1_apellido2_5Junio

# 2. Primer commit — después de P0 y P1
git add evaluacion_final_modulo.ipynb
git commit -m "init: cedulas, seed y generacion de datos"
```

---
## P2 — Interpolación B-spline 2D (27 pts)

Usen `RectBivariateSpline` de `scipy.interpolate` para construir un spline bicúbico sobre los datos `(x_raw, y_raw, Z_data)`.

### P2.1 — Construcción y evaluación (15 pts)

Construyan el spline con `kx=3, ky=3` y evalúenlo en una malla fina de **100×100 puntos** en $[-3,3]\times[-3,3]$.

In [ ]:
# Construir el spline bicúbico
spline = RectBivariateSpline(x_raw, y_raw, Z_data, kx=3, ky=3)

# Crear malla fina de 100x100
n_fina = 100
x_fina = np.linspace(-3, 3, n_fina)
y_fina = np.linspace(-3, 3, n_fina)
X_fina, Y_fina = np.meshgrid(x_fina, y_fina, indexing='ij')

# Evaluar el spline en la malla fina
Z_spline = spline(x_fina, y_fina)


### P2.2 — Figura comparativa (15 pts)

Produzcan una figura con **dos paneles side-by-side** (`subplot` 1×2):
- Panel izquierdo: `Z_data` en la malla gruesa (superficie en 3D).
- Panel derecho: superficie interpolada por el spline en la malla fina (3D).

Ambos paneles deben tener: colorbar, ejes etiquetados, y usar el **mismo colormap y los mismos límites de color** (`vmin`/`vmax` iguales en ambos).

In [ ]:
fig = plt.figure(figsize=(14, 6))

# Límites de color compartidos
vmin = min(Z_data.min(), Z_spline.min())
vmax = max(Z_data.max(), Z_spline.max())

# Panel izquierdo: Z_data en malla gruesa
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf1 = ax1.plot_surface(X_raw, Y_raw, Z_data, cmap='viridis', vmin=vmin, vmax=vmax, edgecolor='k')
ax1.set_title('Datos (Malla Gruesa 15x15)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10)

# Panel derecho: Spline en malla fina
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
surf2 = ax2.plot_surface(X_fina, Y_fina, Z_spline, cmap='viridis', vmin=vmin, vmax=vmax)
ax2.set_title('Interpolación Spline (Malla Fina 100x100)')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('z')
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)

plt.tight_layout()
plt.show()


---
## P3 — Ajuste por mínimos cuadrados (20 pts)

Ajusten un **plano** $z = c_0 + c_1 x + c_2 y$ a los 225 puntos de la malla usando mínimos cuadrados. Trabajen con los datos ruidosos `Z_data`.

### P3.1 — Ajuste (10 pts)

Construyan la matriz $A$, resuelvan el sistema normal con `np.linalg.lstsq`, e impriman los coeficientes $c_0, c_1, c_2$ con 4 decimales.

In [ ]:
# Aplanar las mallas para crear los pares (x_i, y_i)
x_flat = X_raw.flatten()
y_flat = Y_raw.flatten()
z_flat = Z_data.flatten()

# Construcción de la matriz A
A = np.column_stack([np.ones_like(x_flat), x_flat, y_flat])

# Resolución por mínimos cuadrados
c, residuals, rank, s = np.linalg.lstsq(A, z_flat, rcond=None)

c0, c1, c2 = c
print(f"Coeficientes del plano ajustado:")
print(f"c0 = {c0:.4f}")
print(f"c1 = {c1:.4f}")
print(f"c2 = {c2:.4f}")

### P3.2 — Residuos y figura (10 pts)

Calculen el mapa de residuos: $R = Z_{\text{data}} - Z_{\text{plano}}$, donde $Z_{\text{plano}}$ es el plano evaluado en la malla gruesa.

Produzcan una figura con **dos paneles**:
- Panel izquierdo: plano ajustado sobre la malla gruesa, con colorbar y ejes etiquetados.
- Panel derecho: mapa de residuos $R$, con colorbar, ejes etiquetados, y colormap **divergente** centrado en cero.

In [ ]:
# Evaluar el plano en la malla gruesa
Z_plano = c0 + c1 * X_raw + c2 * Y_raw

# Mapa de residuos
R = Z_data - Z_plano

fig = plt.figure(figsize=(14, 6))

# Panel izquierdo: Plano ajustado
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf1 = ax1.plot_surface(X_raw, Y_raw, Z_plano, cmap='viridis', edgecolor='k')
ax1.set_title('Plano Ajustado por Mínimos Cuadrados')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('z')
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10)

# Panel derecho: Mapa de residuos (colormap divergente centrado en 0)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
# Limitar el colorbar alrededor de 0 para que sea verdaderamente divergente
abs_max = np.max(np.abs(R))
surf2 = ax2.plot_surface(X_raw, Y_raw, R, cmap='RdBu', vmin=-abs_max, vmax=abs_max, edgecolor='k')
ax2.set_title('Mapa de Residuos (Z_data - Z_plano)')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('Residuo')
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)

plt.tight_layout()
plt.show()


---
**Recordatorio del flujo:**

```bash
# 1. Crear y moverse a su rama
git checkout -b apellido1_apellido2_5Junio

# 2. Primer commit — después de P0 y P1
git add evaluacion_final_modulo.ipynb
git commit -m "init: cedulas, seed y generacion de datos"

# 3. Segundo commit — después de P2 y P3
git add evaluacion_final_modulo.ipynb
git commit -m "add: interpolacion spline y ajuste LS"
```

---
## P4 — PCA sobre señales sísmicas (15 pts)

Una red de 20 sensores sísmicos registra vibraciones del terreno durante 50 instantes
de tiempo. Las señales están contaminadas con ruido.

In [ ]:
omega1 = 1.0 + (seed % 7) * 0.3
omega2 = 1.0 + (seed % 5) * 0.4

m, p = 20, 50
t = np.linspace(0, 2*np.pi, p)

rng2 = np.random.default_rng(seed + 1)
a = rng2.uniform(0.5, 2.0, m)
b = rng2.uniform(0.5, 2.0, m)

X = np.outer(a, np.cos(omega1 * t)) + np.outer(b, np.sin(omega2 * t))
X += rng.normal(0, 0.3, X.shape)

print(f"Señales: {m} sensores x {p} instantes")
print(f"omega1={omega1:.2f}, omega2={omega2:.2f}")

### P4.1 — Cálculo de componentes principales (7 pts)

Centren las filas de `X` y calculen la SVD con `np.linalg.svd`.

Produzcan una figura con **dos paneles**:
- Panel izquierdo: varianza explicada por cada componente (gráfico de barras).
- Panel derecho: varianza acumulada vs k (puntos + línea), con una línea
  horizontal punteada en el 85%.

Ambos paneles con ejes etiquetados y título.

### P4.2 — Proyección en el espacio PCA (8 pts)

Calculen los scores proyectando las 20 señales centradas sobre los primeros k
componentes que explique más del 89% de la varianza (2 o 3).

Produzcan un scatter de los 20 sensores en el espacio de los primeros k PCs:
- Si k=2: scatter PC1 vs PC2, coloreado por índice de sensor, con colorbar
  y etiquetas de sensor en cada punto.
- Si k=3: tres paneles PC1 vs PC2, PC1 vs PC3, PC2 vs PC3.

En todos los casos: ejes etiquetados con el nombre del componente (`"PC1"`, `"PC2"`...),
líneas en cero, grid, título con el porcentaje de varianza acumulada capturada.

---
## GitHub (10 pts)

Esta sección se califica directamente desde el PR. No hay celdas adicionales.

**Recordatorio del flujo:**

```bash
# 1. Crear y moverse a su rama
git checkout -b apellido1_apellido2_5Junio

# 2. Primer commit — después de P0 y P1
git add evaluacion_final_modulo.ipynb
git commit -m "init: cedulas, seed y generacion de datos"

# 3. Segundo commit — después de P2 y P3
git add evaluacion_final_modulo.ipynb
git commit -m "add: interpolacion spline y ajuste LS"

# 4. Tercer commit — después de P4 y todos los análisis escritos
git add evaluacion_final_modulo.ipynb
git commit -m "add: PCA y analisis escritos"

# 5. Push y abrir PR
git push origin apellido1_apellido2_5Junio
```

Luego en GitHub: **New Pull Request** → base: `main` ← compare: `apellido1_apellido2_5Junio`.

Título del PR: `Evaluación [Apellido1] [Apellido2]`
